In [ ]:
# Colab bootstrap —— 在 Colab 上第一件事就是跑這格。本機開發時它會自動跳過安裝。
#
# --no-deps        Colab 預裝的 torch 是對著它自己的 CUDA 編的，不能讓 pip 重裝
# --force-reinstall 版本號沒變時 pip 會跳過安裝，推了修正之後要靠它抓到新版
#
# 重裝之後舊模組還留在 sys.modules 裡，**要重啟 kernel** 才吃得到新版。
#
# 用 subprocess 而不是 %pip：這樣同一格在本機與 Colab 都能原樣執行。
import importlib
import subprocess
import sys

REPO = "git+https://github.com/318amne-Sia/tfn-pytorch.git@main"

# 用 try/import 而不是 importlib.util.find_spec("google.colab")：後者在沒有
# google 這個套件的環境（例如本機）會直接拋 ModuleNotFoundError，不是回 None。
try:
    import google.colab  # noqa: F401

    in_colab = True
except ImportError:
    in_colab = False

if in_colab:
    # 裝之前先看 tfn 是不是已經被 import 過。__version__ 是寫死在原始碼裡的
    # 字串，印它分不出「剛裝好的新版」與「還留在 sys.modules 的舊版」——
    # 這個檢查才分得出，而且分不出來的話你會抱著舊程式碼一路跑下去。
    already_imported = "tfn" in sys.modules

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-deps", REPO],
        check=True,
    )
    importlib.invalidate_caches()  # 讓剛寫進 site-packages 的檔案馬上被看見

    if already_imported:
        raise RuntimeError(
            "tfn 在這次安裝之前就已經被 import 過，剛裝的版本不會生效。"
            "請重啟 kernel（執行階段 → 重新啟動工作階段）之後再跑這格。"
        )

# 這個 import 刻意放在安裝之後（E402）：Colab 上要先裝好才 import 得到
import tfn  # noqa: E402

# 印出路徑而不只是版本號：本機跑的是工作目錄裡那份，Colab 跑的是 site-packages
print("tfn", tfn.__version__, "來自", tfn.__file__)

# 實驗二之一：牛頓重力

論文 [Tensor Field Networks](https://arxiv.org/abs/1802.08219) §5.2 的前半。

給一堆隨機點和它們的質量，問**每個點受到多大的重力加速度**。答案是一支向量，
所以網路型別是 `0 → 1`：餵進去一個數字，吐出來一支箭頭。

跟實驗一最大的不同不在網路，在**驗收方式**。實驗一有「測試準確率 100%」這種
硬數字；這裡沒有準確率可看——整個網路只有**一條徑向函數**可學，所以成果是
把那條函數整條畫出來，看它是不是自己長成了課本的 $-1/r^2$。

論文要展示的不是「網路學得會」，而是「學到的東西攤開來看，跟物理公式對得上」。

In [ ]:
import numpy as np
import torch

from tfn import gravity
from tfn.utils import normalized_rmse, random_rotation_matrix

SEED = 0

points, masses = gravity.random_points_and_masses(SEED)
print(f"這一組有 {len(points)} 個點")
for p, m in zip(points.tolist(), masses.tolist(), strict=True):
    print(f"  ({p[0]:+.2f}, {p[1]:+.2f}, {p[2]:+.2f})   質量 {m:.2f}")

## 沒有資料集

每一步都現場亂生一組新的點雲，正確答案用物理公式直接算。沒有 epoch、沒有
train/test 切分，也沒有過擬合的空間——資料無限多，而網路只有一條曲線可學。

點的生法有一個關鍵細節：先抽 2–10 個候選點，再把**彼此距離小於 0.5 的剔掉**。
因為目標是 $1/r^2$，兩點靠太近時正確答案會爆炸，訓練會被那幾個極端樣本帶走。

副作用是每一步的點數是浮動的，偶爾甚至只剩 1 個點（那一步沒有梯度，是預期行為）。

In [ ]:
rng = np.random.default_rng(0)
counts, pairs = [], []
for _ in range(5000):
    pts, _ = gravity.random_points_and_masses(rng)
    counts.append(len(pts))
    if len(pts) >= 2:
        distances = torch.cdist(pts, pts)
        pairs.append(distances[~torch.eye(len(pts), dtype=torch.bool)])

counts = np.array(counts)
pairs = torch.cat(pairs)

print(f"剔除後的點數：平均 {counts.mean():.2f}，範圍 {counts.min()}–{counts.max()}")
print(f"只剩不到 2 個點（該步沒有梯度）的比例：{(counts < 2).mean():.2%}")
print()
print(f"點對距離中位數：{pairs.median():.2f}")
print(
    f"超過 RBF 中心上限 {gravity.RBF_HIGH} 的比例：{(pairs > gravity.RBF_HIGH).float().mean():.1%}"
)
print("→ 所以學到的曲線只在 [0.5, 2.0] 這一段有意義。更遠的地方 RBF 響應全是 0，")
print("   網路分辨不出距離，一律吐同一個常數。")

## 等變性是結構帶來的，不是訓練來的

先驗一件事：**還沒訓練**的模型，把座標轉一個角度，輸出的加速度就跟著轉同一個
角度。如果這裡不成立，後面訓練得再漂亮都不算數。

解析解自己也必須是等變的——否則就是拿一組會破壞等變性的標準答案，去教一個
等變的網路。

In [ ]:
torch.manual_seed(SEED)
untrained = gravity.GravityModel()

pts, ms = gravity.random_points_and_masses(1)
rotation = random_rotation_matrix(7)

with torch.no_grad():
    model_gap = (untrained(pts @ rotation.T, ms) - untrained(pts, ms) @ rotation.T).abs().max()

truth_rotated = gravity.accelerations(pts @ rotation.T, ms)
truth_gap = (truth_rotated - gravity.accelerations(pts, ms) @ rotation.T).abs().max()

print(f"未訓練網路   旋轉後 vs 直接轉輸出，最大偏差：{model_gap:.2e}")
print(f"解析解       旋轉後 vs 直接轉輸出，最大偏差：{truth_gap:.2e}")

## 訓練

單層、一條 L=1 路徑、一個通道。Adam `lr = 1e-3`，1001 步，一次一個樣本。
本機 CPU 不到一秒——這個實驗的點雲只有個位數個點，丟 GPU 只會被 kernel launch
開銷拖慢。

loss 是**差的平方和除以 2**（對齊 TF1 的 `tf.nn.l2_loss`），而且涵蓋**全部的點**
——加速度在每個點上同時都有定義，不必先挑一個中心。

In [ ]:
torch.manual_seed(SEED)
model = gravity.GravityModel()
history = gravity.train(model, steps=gravity.TRAIN_STEPS, rng=SEED)

for step in range(0, gravity.TRAIN_STEPS, 200):
    window = history[max(0, step - 49) : step + 1]
    print(f"step {step:5d}   最近 {len(window):2d} 步平均 loss {np.mean(window):8.3f}")

print()
print(f"整個網路的參數量：{sum(p.numel() for p in model.parameters())}")
print("（兩層 MLP：30×30 + 30 + 1×30 + 1。全部的學習都發生在這裡。）")

## 驗收：把那條徑向函數畫出來

這就是論文 §5.2 的成果。沒有人告訴網路 $1/r^2$——它只看過一堆點的座標、質量、
和加速度，反平方律是從那裡面自己長出來的。

陰影區是**網路真的看過資料的距離範圍**。左邊界是最小點距（更近的點對根本不會
被生出來），右邊界是 RBF 中心的上限。兩邊之外曲線長歪了不算錯。

In [ ]:
# 圖上的文字一律用英文：Colab 的 matplotlib 預設字型沒有中文字符，
# 寫中文會整排變成豆腐框。
import matplotlib.pyplot as plt

distances = torch.linspace(0.05, 2.6, 400)

fig, ax = plt.subplots(figsize=(7, 4.2))
ax.axvspan(gravity.MIN_SEPARATION, gravity.RBF_HIGH, color="#2f7d7b", alpha=0.08)
ax.plot(
    distances,
    gravity.analytic_radial(distances),
    "--",
    lw=2.5,
    color="#c0504d",
    label=r"analytic  $-1/r^2$",
)
ax.plot(
    distances, model.radial_curve(distances), lw=2, color="#2f7d7b", label="learned radial function"
)

ax.axvline(gravity.MIN_SEPARATION, color="#555", ls=":", lw=1.4)
ax.axvline(gravity.RBF_HIGH, color="#555", ls=":", lw=1.4)
ax.annotate(
    "min separation\n0.5",
    xy=(gravity.MIN_SEPARATION, -5.2),
    xytext=(0.6, -5.2),
    fontsize=9,
    color="#333",
    va="center",
)
ax.annotate(
    "RBF centres end\n2.0",
    xy=(gravity.RBF_HIGH, -5.2),
    xytext=(1.93, -5.2),
    fontsize=9,
    color="#333",
    va="center",
    ha="right",
)
ax.text(1.25, 0.45, "comparison range", fontsize=9, color="#2f7d7b", ha="center")

ax.set_xlim(0, 2.6)
ax.set_ylim(-8, 1)
ax.set_xlabel("pair distance  r")
ax.set_ylabel("radial function output")
ax.set_title("Newtonian gravity: the network recovers the inverse square law")
ax.legend(loc="lower right")
ax.grid(alpha=0.15)
plt.show()

## 把「疊得上」變成一個數字

肉眼看圖不能當測試，所以用 normalized RMSE：$\mathrm{RMS}(\text{學到的}-\text{解析解})\,/\,\mathrm{RMS}(\text{解析解})$。

取比值而不是絕對誤差，是因為三條待驗收的曲線量級差很多；先各自取 RMS 再相除
而不是逐點相除，是因為 $-1/r^2$ 在近距離發散、逐點算會在區間端點失控。

順帶一個好性質：**整條曲線符號反轉剛好得到 2.0**。CG 係數弄反的症狀就是
「形狀全對、只差一個負號」，而 loss 照樣降得下去——那是這個專案最危險的一類
bug，任何合理的門檻都擋得住它。

In [ ]:
fit = torch.linspace(gravity.MIN_SEPARATION, gravity.RBF_HIGH, 50)
error = normalized_rmse(model.radial_curve(fit), gravity.analytic_radial(fit)).item()
vloss = gravity.validation_loss(model, samples=gravity.VALIDATION_SAMPLES, rng=1)

torch.manual_seed(SEED)
fresh = gravity.GravityModel()
fresh_error = normalized_rmse(fresh.radial_curve(fit), gravity.analytic_radial(fit)).item()

print(f"nRMSE（訓練後）        {error:8.4f}   門檻 {gravity.RADIAL_NRMSE_CEILING}")
print(f"nRMSE（未訓練對照組）  {fresh_error:8.4f}")
print(f"nRMSE（符號整條反轉）  {2.0:8.4f}")
print()
print(f"validation loss        {vloss:8.3f}   門檻 {gravity.VALIDATION_LOSS_CEILING}")
print("  （驗證用最多 50 個點，訓練只到 10——同一條徑向函數換個點數照樣成立，")
print("    這是一個不花錢的泛化檢查）")